# 07 – Spatial Durbin Model (SDM)

**Pipeline stage:** Feature Selection → **Spatial Econometrics (SDM)**

Notebook này sử dụng tập biến đã được chọn lọc từ các bước BMA và LASSO (Notebook 06) để xây dựng mô hình **Spatial Durbin Model (SDM)** cho dữ liệu di cư (Gravity Model).

Do dữ liệu ở dạng dyadic (cặp quốc gia gốc - đích) và dạng panel (nhiều năm), việc áp dụng mô hình không gian đòi hỏi:
1. **Ma trận trọng số không gian (W)**: Dựa trên khoảng cách nghịch đảo hoặc chung biên giới.
2. **Biến trễ không gian (Spatial Lags - WX, WY)**: Tính toán sự lan tỏa không gian tại nước gốc và nước đích (Origin/Destination spatial lags).
3. **Ước lượng**: Mô hình SDM bao gồm biến phụ thuộc trễ không gian (WY) và các biến độc lập trễ không gian (WX).

## 0. Import thư viện

In [17]:
import os, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

os.makedirs('../output/figures', exist_ok=True)
os.makedirs('../output/tables',  exist_ok=True)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 30)
plt.rcParams.update({'figure.dpi':120,'font.size':11})
SEED = 42
np.random.seed(SEED)
print('✅ Import OK')

✅ Import OK


## 1. Load dữ liệu và tập biến đã được chọn (từ BMA & LASSO)

In [18]:
# 13 biến được đồng thuận chọn bởi BMA, LASSO và ElasticNet từ Notebook 06
SELECTED_VARS = [
    'ln_gdppc_org', 'ln_gdppc_des', 'ln_pop_org', 'ln_pop_des',
    'ln_dist', 'comlang_off', 'contig', 'colony', 'inflation_org',
    'inflation_des', 'ln_co2_org', 'ln_disaster_org', 'ln_disaster_des'
]
DEP_VAR = 'ln_migration'

df_raw = pd.read_csv('../data/processed/migration_data_processed_2000_2022.csv', low_memory=False)

def apply_fixes(df):
    d = df.copy()
    d = d[d['migration'] > 0].copy()
    if 'year' in d.columns:
        d = d[d['year'].between(2000, 2022)].copy()
    d['ln_migration'] = np.log(d['migration'])
    for col in ['ln_co2_org','ln_pm25_org','ln_bandwidth_org']:
        if col in d.columns:
            d.loc[d[col] < -20, col] = np.nan
            d.loc[d[col] == 0,  col] = np.nan
    if 'ln_internet_use_org' in d.columns:
        d.loc[d['ln_internet_use_org'] == 0, 'ln_internet_use_org'] = np.nan
    return d

df = apply_fixes(df_raw)

# Loại bỏ missing values trên tập biến đã chọn
# df_sdm = df[SELECTED_VARS + [DEP_VAR, 'iso_org', 'iso_des', 'year', 'pair_id']].dropna().copy()
df_sdm = df[SELECTED_VARS + [DEP_VAR, 'iso_o', 'iso_d', 'year', 'pair_id']].dropna().copy()
print(f'Shape data SDM: {df_sdm.shape}')

Shape data SDM: (8360, 18)


In [30]:
cols = list(df_sdm.columns)

for i in range(len(cols)):
    if "iso_o" in cols[i]:
        cols[i] = "iso_org"
    if "iso_d" in cols[i]:
        cols[i] = "iso_des"

cols
df_sdm.columns = cols
df_sdm


,ln_gdppc_org,ln_gdppc_des,ln_pop_org,ln_pop_des,ln_dist,comlang_off,contig,colony,inflation_org,inflation_des,ln_co2_org,ln_disaster_org,ln_disaster_des,ln_migration,iso_org,iso_des,year,pair_id
0,10.1865,10.2875,16.7615,17.2393,9.6542,1,0,0,4.4574,2.7194,2.9230,1.6094,0.6931,9.6697,AUS,CAN,2000,AUS_CAN
1,10.2272,10.3166,16.7743,17.2502,9.6542,1,0,0,4.4071,2.5251,2.9284,2.0794,0.6931,9.6942,AUS,CAN,2001,AUS_CAN
2,10.2762,10.3406,16.7857,17.2610,9.6542,1,0,0,2.9816,2.2584,2.9396,1.3863,0.6931,9.7187,AUS,CAN,2002,AUS_CAN
3,10.3130,10.3844,16.7972,17.2700,9.6542,1,0,0,2.7326,2.7586,2.9267,2.3026,1.9459,9.7432,AUS,CAN,2003,AUS_CAN
4,10.3661,10.4320,16.8079,17.2793,9.6542,1,0,0,2.3433,1.8573,2.9529,1.9459,1.6094,9.7677,AUS,CAN,2004,AUS_CAN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355,9.2113,11.0489,18.3823,19.6101,9.4849,0,0,0,3.5396,2.4426,1.0817,2.0794,2.7081,14.1358,VNM,USA,2018,VNM_USA
8356,9.3083,11.0856,18.3920,19.6153,9.4849,0,0,0,2.7958,1.8122,1.2493,2.0794,2.9957,14.1426,VNM,USA,2019,VNM_USA
8357,9.3596,11.0729,18.4013,19.6194,9.4849,0,0,0,3.2209,1.2336,1.2780,2.4849,3.0910,14.1493,VNM,USA,2020,VNM_USA
8358,9.3967,11.1748,18.4100,19.6209,9.4849,0,0,0,1.8347,4.6979,1.2377,2.1972,3.5553,14.1561,VNM,USA,2021,VNM_USA


## 2. Xây dựng ma trận trọng số không gian (Spatial Weight Matrix - W)

Đối với dữ liệu di cư, ma trận W có thể được xây ở cấp độ quốc gia, sau đó áp dụng vào tính độ trễ không gian (spatial lags). Chúng ta sẽ dùng trọng số dựa trên nghịch đảo khoảng cách (Inverse Distance).

In [31]:
# Xây dựng danh sách quốc gia
countries = sorted(list(set(df_sdm['iso_org'].unique()) | set(df_sdm['iso_des'].unique())))
country_to_idx = {c: i for i, c in enumerate(countries)}
n_countries = len(countries)

# Khởi tạo ma trận khoảng cách
dist_matrix = np.zeros((n_countries, n_countries))

# Lấy khoảng cách trung bình giữa các cặp vì dist_matrix cần đối xứng
pair_dists = df_sdm.groupby(['iso_org', 'iso_des'])['ln_dist'].mean().reset_index()

for _, row in pair_dists.iterrows():
    i = country_to_idx[row['iso_org']]
    j = country_to_idx[row['iso_des']]
    dist_matrix[i, j] = row['ln_dist']
    dist_matrix[j, i] = row['ln_dist'] # Giả định đối xứng

# Tính ma trận nghịch đảo khoảng cách W (1 / khoảng cách, với diagonal = 0)
W = np.zeros_like(dist_matrix)
mask = dist_matrix > 0
W[mask] = 1 / np.exp(dist_matrix[mask]) # Chuyển từ ln_dist về dist và lấy nghịch đảo
np.fill_diagonal(W, 0)

# Chuẩn hóa ma trận W (Row-standardized)
row_sums = W.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1 # Tránh chia cho 0
W_norm = W / row_sums

print(f"Ma trận trọng số W_norm: {W_norm.shape}")

Ma trận trọng số W_norm: (26, 26)


## 3. Tạo các biến trễ không gian (Spatial Lags - WX)

Tính toán $WX_{org}$ và $WX_{des}$ để phản ánh tác động từ các đặc điểm của quốc gia láng giềng. Ví dụ, lạm phát ở các quốc gia láng giềng có thể ảnh hưởng đến quyết định di cư.

In [32]:
# Cụ thể ta tính toán trễ không gian cho các biến có ý nghĩa nhất định
W_VARS_ORG = ['ln_gdppc_org', 'ln_pop_org', 'inflation_org', 'ln_co2_org', 'ln_disaster_org']
W_VARS_DES = ['ln_gdppc_des', 'ln_pop_des', 'inflation_des', 'ln_disaster_des']

# Tạo dataframe chứa vector đặc điểm theo từng năm cho mỗi quốc gia
def compute_spatial_lags(df, w_matrix, var_cols, role):
    # Lấy features theo year và iso
    features = df.groupby(['year', f'iso_{role}'])[var_cols].mean().reset_index()
    
    lags = []
    for year, group in features.groupby('year'):
        # Vector X cho năm year
        X = np.zeros((n_countries, len(var_cols)))
        for _, row in group.iterrows():
            idx = country_to_idx[row[f'iso_{role}']]
            X[idx, :] = row[var_cols].values
            
        # Tính W * X
        WX = w_matrix.dot(X)
        
        # Lưu kết quả
        for i, c in enumerate(countries):
            record = {'year': year, f'iso_{role}': c}
            for j, v in enumerate(var_cols):
                record[f'W_{v}'] = WX[i, j]
            lags.append(record)
            
    return pd.DataFrame(lags)

lag_org = compute_spatial_lags(df_sdm, W_norm, W_VARS_ORG, 'org')
lag_des = compute_spatial_lags(df_sdm, W_norm, W_VARS_DES, 'des')

# Merge lại vào df_sdm
df_sdm = df_sdm.merge(lag_org, on=['year', 'iso_org'], how='left')
df_sdm = df_sdm.merge(lag_des, on=['year', 'iso_des'], how='left')

# Drop NA cho các dòng không tính được lag
df_sdm = df_sdm.dropna()
print(f'Shape data sau khi thêm WX: {df_sdm.shape}')

Shape data sau khi thêm WX: (8360, 27)


## 4. Mô hình SLX (Spatially Lagged X) / SDM

Ước lượng mô hình gravity nâng cao với phân tích trễ không gian.

In [33]:
# Danh sách biến độc lập: SELECTED_VARS + WX
WX_COLS = [f'W_{v}' for v in W_VARS_ORG] + [f'W_{v}' for v in W_VARS_DES]
ALL_EXOG = SELECTED_VARS + WX_COLS

X = sm.add_constant(df_sdm[ALL_EXOG])
y = df_sdm[DEP_VAR]

# Ước lượng OLS (HC3 Robust) - tương đương với SLX
slx_model = sm.OLS(y, X).fit(cov_type='HC3')

print("=== MÔ HÌNH SPATIAL DURBIN / SLX ===")
print(f'N = {int(slx_model.nobs):,} | R² = {slx_model.rsquared:.4f} | R²_adj = {slx_model.rsquared_adj:.4f}')
print(slx_model.summary().tables[1])

=== MÔ HÌNH SPATIAL DURBIN / SLX ===
N = 8,360 | R² = 0.5715 | R²_adj = 0.5703
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -13.2113      1.206    -10.957      0.000     -15.574     -10.848
ln_gdppc_org          0.0979      0.044      2.220      0.026       0.011       0.184
ln_gdppc_des          1.5026      0.040     37.887      0.000       1.425       1.580
ln_pop_org            0.6005      0.020     29.884      0.000       0.561       0.640
ln_pop_des            0.5646      0.020     27.627      0.000       0.525       0.605
ln_dist              -0.4796      0.018    -26.025      0.000      -0.516      -0.443
comlang_off           0.9849      0.057     17.202      0.000       0.873       1.097
contig                1.3682      0.069     19.701      0.000       1.232       1.504
colony                1.2535      0.073     17.091      0.000

## 5. Kết luận & Tổng kết

- Việc thêm các biến trễ không gian WX giúp làm rõ thêm các yếu tố từ láng giềng tác động như thế nào đến dòng di cư.
- Mô hình có R-squared tốt, cho thấy sự phù hợp hợp lý.
- Các biến đã được lựa chọn qua BMA & LASSO tiếp tục duy trì sự chắc chắn trong các cấu hình nâng cao.